## Outlier Detection and Data Quality

### Import Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

### Define File Paths

In [2]:


PROJECT_DIR = Path.cwd()

INPUT_DIR = (
    PROJECT_DIR
    / "cleaned_data"
    / "feature_engineering"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "cleaned_data"
    / "outlier_detection"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Sold dataset
SOLD_INPUT_FILE = (
    INPUT_DIR
    / "sold_residential_features_enriched.csv"
)

# List dataset
LIST_INPUT_FILE = (
    INPUT_DIR
    / "list_residential_features_enriched.csv"
)

# Confirm dataset is exist
print("Sold input file exists:", SOLD_INPUT_FILE.exists())
print("List input file exists:", LIST_INPUT_FILE.exists())

Sold input file exists: True
List input file exists: True


### Load Sold & List Dataset

In [3]:
sold_df = pd.read_csv(
    SOLD_INPUT_FILE,
    low_memory=False
)

# Avoid naming the List dataset "list" because "list" is a built-in Python data type.
# Use list_df instead
list_df = pd.read_csv(
    LIST_INPUT_FILE,
    low_memory=False
)

print("Sold shape:", sold_df.shape)
print("List shape:", list_df.shape)

Sold shape: (447769, 99)
List shape: (615316, 83)


In [4]:
# Create copies of the Sold and List datasets so the original loaded datasets remain unchanged.
sold_flagged = sold_df.copy()
list_flagged = list_df.copy()

sold_iqr_columns = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

list_iqr_columns = [
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

print("Sold working shape:", sold_flagged.shape)
print("List working shape:", list_flagged.shape)

Sold working shape: (447769, 99)
List working shape: (615316, 83)


### Define the IQR Outlier Detection Function

Calculate the IQR boundaries and create a separate outlier flag for each numeric field.


In [5]:
def add_iqr_outlier_flag(
    df,
    column,
    flag_column,
    valid_mask,
    dataset_name,
    multiplier=1.5
):
    """
    Calculate IQR boundaries and add an outlier flag.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset receiving the outlier flag.
    column : str
        Numeric field used for IQR detection.
    flag_column : str
        Name of the new outlier flag.
    valid_mask : pandas.Series
        Condition identifying valid values.
    dataset_name : str
        Dataset label used in the summary.
    multiplier : float
        IQR multiplier. The standard value is 1.5.

    Returns
    -------
    dict
        Summary of the calculated IQR thresholds.
    """

    valid_values = df.loc[
        valid_mask
        & df[column].notna(),
        column
    ]

    if valid_values.empty:
        raise ValueError(
            f"No valid values are available for {column}."
        )

    q1 = valid_values.quantile(0.25)
    median = valid_values.quantile(0.50)
    q3 = valid_values.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - multiplier * iqr
    upper_bound = q3 + multiplier * iqr

    df[flag_column] = (
        valid_mask
        & df[column].notna()
        & (
            df[column].lt(lower_bound)
            | df[column].gt(upper_bound)
        )
    )

    outlier_count = int(
        df[flag_column].sum()
    )

    valid_count = int(
        valid_values.shape[0]
    )

    return {
        "dataset": dataset_name,
        "column": column,
        "valid_count": valid_count,
        "q1": q1,
        "median": median,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percentage": (
            outlier_count
            / valid_count
            * 100
        )
    }

### Apply IQR Detection to the Sold Dataset

Apply IQR detection to ClosePrice, LivingArea, and DaysOnMarket using valid business-rule values.

In [6]:
sold_iqr_results = []

sold_iqr_results.append(
    add_iqr_outlier_flag(
        df=sold_flagged,
        column="ClosePrice",
        flag_column="close_price_iqr_outlier_flag",
        valid_mask=sold_flagged["ClosePrice"].gt(0),
        dataset_name="Sold"
    )
)

sold_iqr_results.append(
    add_iqr_outlier_flag(
        df=sold_flagged,
        column="LivingArea",
        flag_column="living_area_iqr_outlier_flag",
        valid_mask=sold_flagged["LivingArea"].gt(0),
        dataset_name="Sold"
    )
)

sold_iqr_results.append(
    add_iqr_outlier_flag(
        df=sold_flagged,
        column="DaysOnMarket",
        flag_column="days_on_market_iqr_outlier_flag",
        valid_mask=sold_flagged["DaysOnMarket"].ge(0),
        dataset_name="Sold"
    )
)

sold_iqr_summary = pd.DataFrame(
    sold_iqr_results
)

display(sold_iqr_summary)

,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,Sold,ClosePrice,447767,575000.0,825000.0,1300000.0,725000.0,-512500.0,2387500.0,33477,7.476433
1,Sold,LivingArea,447516,1248.0,1646.0,2224.0,976.0,-216.0,3688.0,19568,4.372581
2,Sold,DaysOnMarket,447769,8.0,18.0,48.0,40.0,-52.0,108.0,34144,7.625360


### Apply IQR Detection to the List Dataset

Apply IQR detection to ListPrice, LivingArea, and DaysOnMarket using valid business-rule values.

In [7]:
list_iqr_results = []

list_iqr_results.append(
    add_iqr_outlier_flag(
        df=list_flagged,
        column="ListPrice",
        flag_column="list_price_iqr_outlier_flag",
        valid_mask=list_flagged["ListPrice"].gt(0),
        dataset_name="List"
    )
)

list_iqr_results.append(
    add_iqr_outlier_flag(
        df=list_flagged,
        column="LivingArea",
        flag_column="living_area_iqr_outlier_flag",
        valid_mask=list_flagged["LivingArea"].gt(0),
        dataset_name="List"
    )
)

list_iqr_results.append(
    add_iqr_outlier_flag(
        df=list_flagged,
        column="DaysOnMarket",
        flag_column="days_on_market_iqr_outlier_flag",
        valid_mask=list_flagged["DaysOnMarket"].ge(0),
        dataset_name="List"
    )
)

list_iqr_summary = pd.DataFrame(
    list_iqr_results
)

display(list_iqr_summary)

,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,List,ListPrice,615316,581000.0,849000.0,1385000.0,804000.0,-625000.0,2591000.0,51489,8.367896
1,List,LivingArea,614693,1248.0,1673.0,2304.0,1056.0,-336.0,3888.0,30305,4.930103
2,List,DaysOnMarket,615316,5.0,11.0,22.0,17.0,-20.5,47.5,47782,7.765441


In [8]:
# Create Overall Outlier Flags
# Combine the individual IQR flags to identify records containing at least one statistical outlier.

sold_iqr_flag_columns = [
    "close_price_iqr_outlier_flag",
    "living_area_iqr_outlier_flag",
    "days_on_market_iqr_outlier_flag"
]

list_iqr_flag_columns = [
    "list_price_iqr_outlier_flag",
    "living_area_iqr_outlier_flag",
    "days_on_market_iqr_outlier_flag"
]

# Create an overall Sold outlier flag.
# The flag is True when at least one of the selected
# Sold IQR outlier flags is True for that record.
sold_flagged["any_iqr_outlier_flag"] = (
    sold_flagged[sold_iqr_flag_columns]
    .any(axis=1)
)

# Create an overall List outlier flag.
# The flag is True when at least one of the selected
# List IQR outlier flags is True for that record.
list_flagged["any_iqr_outlier_flag"] = (
    list_flagged[list_iqr_flag_columns]
    .any(axis=1)
)

### Validate Outlier Detection Results

In [9]:
print("Sold IQR outlier counts:")

display(
    sold_flagged[
        sold_iqr_flag_columns
        + ["any_iqr_outlier_flag"]
    ]
    .sum()
    .to_frame("count")
)

print("List IQR outlier counts:")

display(
    list_flagged[
        list_iqr_flag_columns
        + ["any_iqr_outlier_flag"]
    ]
    .sum()
    .to_frame("count")
)

Sold IQR outlier counts:


,count
close_price_iqr_outlier_flag,33477
living_area_iqr_outlier_flag,19568
days_on_market_iqr_outlier_flag,34144
any_iqr_outlier_flag,70296


List IQR outlier counts:


,count
list_price_iqr_outlier_flag,51489
living_area_iqr_outlier_flag,30305
days_on_market_iqr_outlier_flag,47782
any_iqr_outlier_flag,102567


In [10]:
display(sold_iqr_summary)
display(list_iqr_summary)

,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,Sold,ClosePrice,447767,575000.0,825000.0,1300000.0,725000.0,-512500.0,2387500.0,33477,7.476433
1,Sold,LivingArea,447516,1248.0,1646.0,2224.0,976.0,-216.0,3688.0,19568,4.372581
2,Sold,DaysOnMarket,447769,8.0,18.0,48.0,40.0,-52.0,108.0,34144,7.625360


,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,List,ListPrice,615316,581000.0,849000.0,1385000.0,804000.0,-625000.0,2591000.0,51489,8.367896
1,List,LivingArea,614693,1248.0,1673.0,2304.0,1056.0,-336.0,3888.0,30305,4.930103
2,List,DaysOnMarket,615316,5.0,11.0,22.0,17.0,-20.5,47.5,47782,7.765441


### Create Full Flagged and Filtered Datasets

Preserve all records in the full flagged datasets and create separate filtered datasets by excluding records with at least one IQR outlier.

In [14]:
# Create full flagged datasets that preserve all records and IQR outlier flags
sold_full_flagged = sold_flagged.copy()
list_full_flagged = list_flagged.copy()

# Create filtered Sold dataset by removing records with any IQR outlier
sold_filtered = sold_full_flagged.loc[
    ~sold_full_flagged["any_iqr_outlier_flag"]
].copy()

# Create filtered List dataset by removing records with any IQR outlier
list_filtered = list_full_flagged.loc[
    ~list_full_flagged["any_iqr_outlier_flag"]
].copy()

print("Sold full flagged shape:", sold_full_flagged.shape)
print("Sold filtered shape:", sold_filtered.shape)

print("List full flagged shape:", list_full_flagged.shape)
print("List filtered shape:", list_filtered.shape)

Sold full flagged shape: (447769, 103)
Sold filtered shape: (377473, 103)
List full flagged shape: (615316, 87)
List filtered shape: (512749, 87)


### Compare Row Counts Before and After Filtering

Compare dataset sizes before and after IQR filtering to measure how many records were removed.

In [15]:
row_count_comparison = pd.DataFrame([
    {
        "dataset": "Sold",
        "rows_before": len(sold_full_flagged),
        "rows_after": len(sold_filtered),
        "rows_removed": (
            len(sold_full_flagged)
            - len(sold_filtered)
        ),
        "removed_percentage": (
            (
                len(sold_full_flagged)
                - len(sold_filtered)
            )
            / len(sold_full_flagged)
            * 100
        )
    },
    {
        "dataset": "List",
        "rows_before": len(list_full_flagged),
        "rows_after": len(list_filtered),
        "rows_removed": (
            len(list_full_flagged)
            - len(list_filtered)
        ),
        "removed_percentage": (
            (
                len(list_full_flagged)
                - len(list_filtered)
            )
            / len(list_full_flagged)
            * 100
        )
    }
])

display(row_count_comparison)

,dataset,rows_before,rows_after,rows_removed,removed_percentage
0,Sold,447769,377473,70296,15.699166
1,List,615316,512749,102567,16.668996


### Compare Median Values Before and After Filtering

Compare median values of the selected numeric fields before and after IQR filtering to evaluate the impact of outlier removal.

In [16]:
median_comparison = []

# Compare Sold median values
for column in sold_iqr_columns:
    median_before = sold_full_flagged[column].median()
    median_after = sold_filtered[column].median()

    median_comparison.append({
        "dataset": "Sold",
        "column": column,
        "median_before": median_before,
        "median_after": median_after,
        "median_difference": (
            median_after - median_before
        )
    })

# Compare List median values
for column in list_iqr_columns:
    median_before = list_full_flagged[column].median()
    median_after = list_filtered[column].median()

    median_comparison.append({
        "dataset": "List",
        "column": column,
        "median_before": median_before,
        "median_after": median_after,
        "median_difference": (
            median_after - median_before
        )
    })

median_comparison = pd.DataFrame(
    median_comparison
)

display(median_comparison)

,dataset,column,median_before,median_after,median_difference
0,Sold,ClosePrice,825000.0,787500.0,-37500.0
1,Sold,LivingArea,1646.0,1572.0,-74.0
2,Sold,DaysOnMarket,18.0,16.0,-2.0
3,List,ListPrice,849000.0,798000.0,-51000.0
4,List,LivingArea,1673.0,1587.0,-86.0
5,List,DaysOnMarket,11.0,9.0,-2.0


### Save Full Flagged and Filtered Datasets

In [12]:
# Define output file paths

SOLD_FLAGGED_OUTPUT = (
    OUTPUT_DIR
    / "sold_full_flagged.csv"
)

SOLD_FILTERED_OUTPUT = (
    OUTPUT_DIR
    / "sold_filtered.csv"
)

LIST_FLAGGED_OUTPUT = (
    OUTPUT_DIR
    / "list_full_flagged.csv"
)

LIST_FILTERED_OUTPUT = (
    OUTPUT_DIR
    / "list_filtered.csv"
)

In [18]:
# Save the full Sold dataset with IQR outlier flags
sold_flagged.to_csv(
    SOLD_FLAGGED_OUTPUT,
    index=False
)

# Save the Sold analysis dataset without IQR outliers
sold_filtered.to_csv(
    SOLD_FILTERED_OUTPUT,
    index=False
)

# Save the full List dataset with IQR outlier flags
list_flagged.to_csv(
    LIST_FLAGGED_OUTPUT,
    index=False
)

# Save the List analysis dataset without IQR outliers
list_filtered.to_csv(
    LIST_FILTERED_OUTPUT,
    index=False
)

all_files_saved = all([
    SOLD_FLAGGED_OUTPUT.exists(),
    SOLD_FILTERED_OUTPUT.exists(),
    LIST_FLAGGED_OUTPUT.exists(),
    LIST_FILTERED_OUTPUT.exists()
])

print(
    "All datasets saved successfully:",
    all_files_saved
)

All datasets saved successfully: True
